# Regional 3D building view-factor graph

This notebook runs the finite-height, terrain-aware OpenView graph pipeline. It first processes two 2 km downtown LA tiles, checks the area-based graph outputs, and then defines the resumable Los Angeles County run for buildings within one mile of the CAL FIRE **Interface** class.

The regional default uses at most three vertical quadrature patches per facade column. This retains finite height and over-building visibility while bounding county-scale cost. Increase `max_vertical_patches` for a convergence test before a final publication run.

In [1]:
import os
from pathlib import Path

# Avoid mixed base/environment PROJ databases on some conda installations.
if os.environ.get('CONDA_PREFIX'):
    os.environ.setdefault('PROJ_LIB', str(Path(os.environ['CONDA_PREFIX']) / 'share' / 'proj'))

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import pyogrio
from pyproj import Transformer

from openviewcore.network import BVF3DGraphConfig, build_bvf_3d_graph
pd.set_option('display.max_columns', 50)

In [2]:
def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the OpenView repository.')

PROJECT_ROOT = find_project_root()
TARGET_CRS = 'EPSG:26911'
LARIAC_PATH = Path(os.environ.get('LARIAC_BUILDINGS', '/Users/adamswietek/Documents/PostDoc/data/raw_la/laraic_files/LARIAC6_Buildings_2020.gdb'))
LARIAC_LAYER = 'LARIAC6_BUILDINGS_2020'
DEM_PATH = Path(os.environ.get('LA_COUNTY_DEM', '/Users/adamswietek/Documents/PostDoc/data/raw_la/dem/USGS_1_3_Arc_Second_LosAngelesCounty.tif'))
WUI_PATH = Path(os.environ.get('CALFIRE_WUI', PROJECT_ROOT / 'notebooks' / 'eda' / 'calfire_wui_la.gpkg'))
OUTPUT_ROOT = Path(os.environ.get('OPENVIEW_BVF_OUTPUT_ROOT', PROJECT_ROOT / 'results'))
for path in (LARIAC_PATH, DEM_PATH, WUI_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

## CAL FIRE Interface selection
Only `WUI_DESC == 'Interface'` is buffered. Intermix and Influence Zone polygons are intentionally excluded.

In [3]:
WUI_BUFFER_M = 1_609.344
WUI_SIMPLIFY_M = 25.0
wui = gpd.read_file(WUI_PATH).to_crs(TARGET_CRS)
wui_interface = wui[wui['WUI_DESC'].astype(str).str.casefold().eq('interface')].copy()
wui_1mi_mask = (wui_interface.geometry
    .simplify(WUI_SIMPLIFY_M, preserve_topology=True)
    .buffer(WUI_BUFFER_M, resolution=4)
    .union_all())
print(f'{len(wui_interface):,} Interface features; mask bounds: {tuple(round(v, 1) for v in wui_1mi_mask.bounds)}')

140 Interface features; mask bounds: (314111.6, 3751662.5, 424954.8, 3799745.6)


## Two-tile production-path test
This is deliberately a dense downtown test. On the development machine the two tiles required about four minutes total and produced roughly 0.77 million visible patch exchanges.

In [4]:
TEST_BOUNDS = (384_000.0, 3_768_000.0, 388_000.0, 3_770_000.0)
TEST_OUTPUT = OUTPUT_ROOT / 'bvf_3d_two_tile_demo'
test_config = BVF3DGraphConfig(
    layer=LARIAC_LAYER, id_column='OBJECTID', source_id_column='BLD_ID',
    target_crs=TARGET_CRS, tile_size_m=2_000, max_distance_m=152.4,
    max_angle_deg=60, height_column='HEIGHT', height_scale_to_m=0.3048,
    elevation_column='ELEV', elevation_scale_to_m=0.3048,
    max_patch_width_m=20, max_patch_height_m=3, max_vertical_patches=3,
    dem_path=str(DEM_PATH), terrain_resolution_m=30,
    use_dem_for_building_elevation=True, workers=1, resume=True,
)
test_result = build_bvf_3d_graph(
    LARIAC_PATH, TEST_OUTPUT, config=test_config, bounds=TEST_BOUNDS, max_tiles=2,
)
test_result

3D BVF tiles:   0%|          | 0/2 [00:00<?, ?tile/s]

tile 192_1884: 25,887 facade columns:   0%|          | 0/52 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [ ]:
edges_3d = pd.read_parquet(test_result.building_edges_path)
metrics_3d = pd.read_parquet(test_result.building_metrics_path)
display(pd.Series({
    'buildings': len(metrics_3d), 'building edges': len(edges_3d),
    'contact edges': int(edges_3d.has_contact.sum()),
    'max BVF': metrics_3d.bvf_all_edges.max(),
    'max conservation excess': max(0.0, metrics_3d.bvf_all_edges.max() - 1.0),
}).to_frame('value'))
display(edges_3d.sort_values('vf_i_to_j', ascending=False).head(10))

In [ ]:
source_crs = pyogrio.read_info(LARIAC_PATH, layer=LARIAC_LAYER)['crs']
source_bbox = Transformer.from_crs(TARGET_CRS, source_crs, always_xy=True).transform_bounds(*TEST_BOUNDS, densify_pts=21)
sample = pyogrio.read_dataframe(LARIAC_PATH, layer=LARIAC_LAYER, bbox=source_bbox, columns=['BLD_ID'], fid_as_index=True).to_crs(TARGET_CRS)
sample['building_id'] = sample.index.astype('int64')
sample = sample.merge(metrics_3d, on='building_id', how='left')
sample = sample.cx[TEST_BOUNDS[0]:TEST_BOUNDS[2], TEST_BOUNDS[1]:TEST_BOUNDS[3]]
fig, ax = plt.subplots(figsize=(12, 7))
sample.plot(ax=ax, column='exchange_total_m2', cmap='magma', scheme='quantiles', edgecolor='0.7', linewidth=0.15, legend=True)
ax.set_title('Two-tile total outgoing 3D exchange [m²]')
ax.set_axis_off(); plt.show()

## Full one-mile Interface run
The mask currently intersects 624 two-kilometre tiles. Start with 4–8 workers and monitor memory. The run is resumable at tile boundaries. Nearby buildings outside the mask remain scene occluders but are not graph nodes. Use a new output directory after changing any geometric or sampling parameter.

In [ ]:
RUN_FULL_WUI = False
FULL_WORKERS = int(os.environ.get('OPENVIEW_BVF_WORKERS', '8'))
FULL_OUTPUT = OUTPUT_ROOT / 'lariac6_wui_interface_1mi_bvf_3d'
full_config = BVF3DGraphConfig(
    layer=LARIAC_LAYER, id_column='OBJECTID', source_id_column='BLD_ID',
    target_crs=TARGET_CRS, tile_size_m=2_000, max_distance_m=152.4,
    max_angle_deg=60, height_column='HEIGHT', height_scale_to_m=0.3048,
    elevation_column='ELEV', elevation_scale_to_m=0.3048,
    max_patch_width_m=20, max_patch_height_m=3, max_vertical_patches=3,
    dem_path=str(DEM_PATH), terrain_resolution_m=30,
    use_dem_for_building_elevation=True, workers=FULL_WORKERS, resume=True,
)
if RUN_FULL_WUI:
    full_result = build_bvf_3d_graph(
        LARIAC_PATH, FULL_OUTPUT, config=full_config, selection_mask=wui_1mi_mask,
    )
    display(pd.read_parquet(full_result.building_metrics_path).describe())
    print(full_result.output_dir)
else:
    print(f'Dry run. Set RUN_FULL_WUI=True to process the 624-tile Interface mask with {FULL_WORKERS} workers.')

## Headless equivalent

```bash
openview-bvf build3d \
  --buildings /path/to/LARIAC6_Buildings_2020.gdb \
  --layer LARIAC6_BUILDINGS_2020 \
  --dem /path/to/USGS_1_3_Arc_Second_LosAngelesCounty.tif \
  --selection notebooks/eda/calfire_wui_la.gpkg \
  --selection-field WUI_DESC --selection-value Interface \
  --selection-buffer 1609.344 --selection-simplify 25 \
  --output /path/to/lariac6_wui_interface_1mi_bvf_3d --workers 8
```

The attribute match is case-insensitive and ensures Intermix and Influence Zone polygons are excluded.